In [14]:
from typing import TypedDict
import uuid

from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import START,END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command

In [15]:
class State(TypedDict):
   """The graph state."""
   some_text: str


In [16]:
def human_node(state: State):
   value = interrupt(
      # Any JSON serializable value to surface to the human.
      # For example, a question or a piece of text or a set of keys in the state
      {
         "text_to_revise": state["some_text"]
      }
   )
   return {
      # Update the state with the human's input
      "some_text": value
   }

In [18]:
def node1(state:State):
    return {"some_text":"node1 it is"}

In [19]:
graph_builder = StateGraph(State)
# Add the human-node to the graph
graph_builder.add_node("human_node", human_node)
graph_builder.add_node("node1",node1)
graph_builder.add_edge(START, "human_node")
graph_builder.add_edge("human_node","node1")
graph_builder.add_edge("node1",END)
# A checkpointer is required for `interrupt` to work.


checkpointer = MemorySaver()
graph = graph_builder.compile(
   checkpointer=checkpointer
)




In [20]:
# Pass a thread ID to the graph to run it.
thread_config = {"configurable": {"thread_id":1}}

In [21]:
for chunk in graph.stream({"some_text": "Original text"}, config=thread_config):
   print(chunk)



{'__interrupt__': (Interrupt(value={'text_to_revise': 'Original text'}, resumable=True, ns=['human_node:f3fd2efe-eb35-49a9-0e5a-76c90884ebbb']),)}


In [35]:
graph.get_state(config=thread_config).values

{'some_text': 'Original text'}

In [12]:
# Resume using Command
for chunk in graph.stream(Command(resume="Edited text"), config=thread_config):
   print(chunk)

{'human_node': {'some_text': 'Edited text'}}
